# AquaVision - Segmenter Train + Test on **Kaggle** (DeepFish + Brackish, unaugmented baseline)

**Kaggle setup:** Accelerator = **GPU T4 x2**, Internet = **On**. Add Data -> attach
**`aquavision-scripts`** and **`aquavision-brackish`**. DeepFish is pulled from Hugging Face in-notebook
(no dataset needed yet). Run via **Save Version -> Save & Run All (Commit)** for a disconnect-proof run.

Inputs are read-only (`/kaggle/input/...`); outputs go to `/kaggle/working/` and show in the **Output** tab.

## 0 - Check attached datasets (confirm the folder names)

In [ ]:
!ls /kaggle/input

## 1 - GPU + install
Kaggle has CUDA torch; only ultralytics + huggingface_hub are added. Never pip-install torch.

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics huggingface_hub
import torch
assert torch.cuda.is_available(), "No CUDA torch - set Accelerator=GPU and do not upgrade torch"
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0))

## 2 - Scripts + config
If step 0 showed different folder names, edit `SCRIPTS` / `RF_BRACKISH` to match.

In [ ]:
import os, shutil, glob
def find_input(name):                                # Kaggle mounts at /kaggle/input/<name> or /kaggle/input/datasets/<user>/<name>
    hits = glob.glob(f"/kaggle/input/{name}") + glob.glob(f"/kaggle/input/*/*/{name}")
    assert hits, f"{name} not attached - Add Input -> {name} (see !ls /kaggle/input)"
    return hits[0]
SCRIPTS     = find_input("aquavision-scripts")
RF_BRACKISH = find_input("aquavision-brackish")      # the Roboflow Brackish dataset
print("SCRIPTS =", SCRIPTS, "| RF_BRACKISH =", RF_BRACKISH)
REPO = "/kaggle/working/aquavision"; os.makedirs(REPO, exist_ok=True)
NEED = {"prepare_data.py": "--size_strata", "turbid_split.py": "lambda _, k=k, v=v", "eval_seg.py": "recall_by_size"}
for f, marker in NEED.items():                        # eval_seg.py = the SAME test script the augmented runs use
    assert os.path.exists(f"{SCRIPTS}/{f}"), f"{f} missing from aquavision-scripts - dataset -> New Version -> add it"
    shutil.copy(f"{SCRIPTS}/{f}", f"{REPO}/{f}")
    assert marker in open(f"{REPO}/{f}").read(), f"{f} in aquavision-scripts is an OLD version - upload the current one"
os.chdir(REPO); print("cwd:", os.getcwd())

DEEPFISH = "/kaggle/working/data/deepfish"           # set precisely after the HF download (next cell)
BRACKISH = "/kaggle/working/data/brackish_frames"    # produced by the converter
WORK = "/kaggle/working/work"
RUNS = "/kaggle/working/runs"
EPOCHS_DET = 100                                     # upper bound; early stopping (PATIENCE) ends it sooner
PATIENCE = 25

## 3 - Get DeepFish (Hugging Face, ~7 GiB, this run)
Prints the tree so you can set `DEEPFISH` to the **Segmentation** subfolder (images + `masks/`).
*Optional later:* Commit this once and save the output as an `aquavision-deepfish` dataset to skip
re-downloading (see CLAUDE.md s8).

In [ ]:
import os, glob, tarfile
os.makedirs("/kaggle/working/data", exist_ok=True)
from huggingface_hub import snapshot_download
snapshot_download("Alzayats/DeepFish", repo_type="dataset", local_dir="/kaggle/working/data/deepfish")
for t in glob.glob("/kaggle/working/data/deepfish/*.tar"):
    tarfile.open(t).extractall("/kaggle/working/data/deepfish"); os.remove(t)
print("---- DeepFish tree (find the Segmentation folder: images + masks/) ----")
!ls -R /kaggle/working/data/deepfish | head -50
# >>> set DEEPFISH to the Segmentation subfolder shown above, e.g.:
DEEPFISH = "/kaggle/working/data/deepfish/DeepFish/Segmentation"
print("DEEPFISH =", DEEPFISH)

## 4 - Convert Brackish (Roboflow) -> single-class 'fish' frames (symlink, fish-only, fast)
Roboflow names end in a hash (`<clip>-0059_jpg.rf.<hash>.jpg`), which hid the video clip from the
group-aware split (every frame became its own group -> neighbouring frames leaked into test). Each
frame is now renamed `<clip>_frame_<NNNN>`, so `prepare_data.py` groups by clip (55 clips contain fish).

In [ ]:
# Convert the attached Roboflow Brackish dataset -> single-class 'fish' frames for prepare_data.
# FAST: symlinks (no file copy) + keeps only fish-containing frames + live progress.
# Files are renamed <clip>_frame_<NNNN> so prepare_data's group_key() splits by video clip (no leakage).
import re, os, shutil, yaml
from collections import Counter
from pathlib import Path
RF = Path(RF_BRACKISH)
BR = Path(BRACKISH)
shutil.rmtree(BR, ignore_errors=True)                 # clear any partial output
BR.mkdir(parents=True, exist_ok=True)
dy = next(RF.rglob("data.yaml"), None)
assert dy is not None, f"no data.yaml under {RF} - is the Brackish dataset attached and named right? (see /kaggle/input)"
names = yaml.safe_load(dy.read_text())["names"]
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
norm = lambda n: re.sub(r"[\s\-]+", "_", str(n).strip().lower())
fish_idx = {i for i, n in enumerate(names) if norm(n) in {"fish", "small_fish", "smallfish"}}
print("classes:", names, "-> fish indices:", sorted(fish_idx), flush=True)

RF_NAME = re.compile(r"^(?P<clip>.+)-(?P<frame>\d+)_(?:jpg|jpeg|png)\.rf\.[0-9a-f]+$", re.I)
imgs = [p for e in ("*.jpg", "*.jpeg", "*.png") for p in RF.rglob(e)]
print(f"{len(imgs)} Brackish images found; converting (symlink, fish-only)...", flush=True)
kept = boxes = dup = 0
clips = Counter()
for k, im in enumerate(imgs, 1):
    m = RF_NAME.match(im.stem)
    assert m, f"unexpected Roboflow file name (cannot find clip/frame): {im.name}"
    lab = im.parent.parent / "labels" / f"{im.stem}.txt"
    if not lab.exists(): lab = im.with_suffix(".txt")
    bx = []
    if lab.exists():
        for line in lab.read_text().splitlines():
            v = line.split()
            if len(v) >= 5 and int(float(v[0])) in fish_idx:
                bx.append("1 " + " ".join(v[1:5]))
    if not bx:                                         # skip frames with no fish (fast + fish-relevant)
        continue
    stem = f"{m['clip']}_frame_{int(m['frame']):04d}"
    dst = BR / f"{stem}{im.suffix.lower()}"
    if dst.exists():                                   # same clip frame exported twice -> keep one
        dup += 1
        continue
    os.symlink(im, dst)                                # instant, no copy
    (BR / f"{stem}.txt").write_text("\n".join(bx))
    kept += 1; boxes += len(bx); clips[m["clip"]] += 1
    if k % 2000 == 0:
        print(f"  {k}/{len(imgs)} scanned, {kept} fish frames kept", flush=True)
print(f"DONE: {kept} Brackish fish frames ({boxes} boxes) from {len(clips)} clips -> {BR}"
      f"  ({dup} duplicate frames skipped)", flush=True)
assert kept, "no fish frames found - check the Brackish dataset is attached"
BRACKISH = str(BR)

## 5 - Prepare detector data (DeepFish + Brackish)
Single-class `fish` YOLO-seg, 70/20/10 group-aware split. No augmentation.
`--size_strata 2` splits clips inside fish-size bins (+ a crowded-clip flag), so train/val/test get the
same mix of small schooling vs large single fish (the unstratified split gave train 44 px vs test 99 px).
The printout under each `[split:...]` line shows fish/img and median fish size per split - they should be similar.

In [ ]:
!python prepare_data.py detector --deepfish "{DEEPFISH}" --brackish "{BRACKISH}" --out "{WORK}/det_baseline" --bg_ratio 1.0 --size_strata 2

# Leakage check: no video clip / sequence may appear in more than one split.
import csv, sys
from collections import defaultdict
sys.path.insert(0, REPO)
from prepare_data import group_key
seen = defaultdict(set)
for r in csv.DictReader(open(f"{WORK}/det_baseline/manifest.csv")):
    seen[(r["dataset"], group_key(r["file"].split("__", 1)[1].rsplit(".", 1)[0]))].add(r["split"])
for ds in sorted({d for d, _ in seen}):
    g = {k: v for k, v in seen.items() if k[0] == ds}
    print(f"[{ds}] {len(g)} groups, {sum(len(v) > 1 for v in g.values())} span >1 split")
leaks = [k for k, v in seen.items() if len(v) > 1]
assert not leaks, f"LEAKAGE: {len(leaks)} groups in more than one split, e.g. {leaks[:3]}"
print("OK: no clip/sequence appears in more than one split")

## 6 - Train detector baseline (augmentation OFF, early stopping)
`patience=25`: stops once validation fitness hasn't improved for 25 epochs (the last run peaked at
epoch 20 and only got worse after). `best.pt` is still the best validation epoch.

In [ ]:
from ultralytics import YOLO
AUG_OFF = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, degrees=0.0, translate=0.0, scale=0.0,
               shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.0,
               mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0, auto_augment=None)
det = YOLO("yolo11s-seg.pt")   # COCO-pretrained; no data augmentation
# exist_ok=True: always write to runs/detector_baseline (no silent detector_baseline-2 folders)
det.train(data=f"{WORK}/det_baseline/data.yaml", epochs=EPOCHS_DET, patience=PATIENCE, imgsz=640, batch=16,
          device=0, workers=2, project=RUNS, name="detector_baseline", exist_ok=True,
          save_period=10, **AUG_OFF)
DET_DIR = str(det.trainer.save_dir); print("run folder:", DET_DIR)

## 7 - Test with `eval_seg.py` (shared with the augmented runs)
Overall + Brackish / DeepFish (Brackish masks are rectangles, DeepFish masks are real) + clean / turbid halves
+ recall by fish size + precision at conf 0.25 -> `test_metrics.json`, plots -> `test_plots/`.

In [ ]:
!python eval_seg.py --weights "{DET_DIR}/weights/best.pt" --det "{WORK}/det_baseline" --out "{DET_DIR}" --project "{RUNS}"

## 8 - Weights + files to download
Everything for this run is in `/kaggle/working/runs/detector_baseline/` (Output tab after a Commit).
The final `best.pt` is ~20 MB; a ~100 MB one is an unfinished checkpoint. The raw downloads
(`/kaggle/working/data`, ~7 GB of DeepFish) are deleted at the end so the Output tab stays small and fast.

In [ ]:
import os, shutil
for f in ["manifest.csv", "data.yaml", "raw_counts.json", "turbidity.csv"]:   # copy the data card next to the run
    if os.path.exists(f"{WORK}/det_baseline/{f}"): shutil.copy(f"{WORK}/det_baseline/{f}", DET_DIR)
for f in ["weights/best.pt", "results.csv", "args.yaml", "test_metrics.json", "manifest.csv", "turbidity.csv",
          "results.png", "test_plots/confusion_matrix.png", "test_plots/MaskPR_curve.png"]:
    p = f"{DET_DIR}/{f}"
    print(f"OK      {p}  ({os.path.getsize(p) / 1e6:.1f} MB)" if os.path.exists(p) else f"MISSING {p}")

shutil.rmtree("/kaggle/working/data", ignore_errors=True)   # raw DeepFish + Brackish symlinks (not needed after prep)
print("removed /kaggle/working/data (raw downloads)")